<a href="https://colab.research.google.com/github/ranispl/FSOI/blob/develop/Satellite_Track_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 22.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [netCDF4]


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone
from netCDF4 import Dataset

# ============================================================
# 35-degree inclination test case for OSSE satellite tracks
# Outputs:
#   - NetCDF: osse_inc35_orbit.nc
#   - CSV:    osse_inc35_orbit.csv
#   - Plot:   osse_inc35_orbit_track.png
# ============================================================

R_EARTH = 6378.137
MU_EARTH = 398600.4418

def deg2rad(x):
    return np.deg2rad(x)

def norm(v):
    return np.linalg.norm(v)

def jd_from_datetime(dt):
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    return 2440587.5 + (dt - epoch).total_seconds() / 86400.0

def rotation_z(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

def rotation_x(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0,  c, -s],
        [0.0,  s,  c]
    ])

def spherical_to_cartesian(lat_deg, lon_deg, radius_km=R_EARTH):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    x = radius_km * np.cos(lat) * np.cos(lon)
    y = radius_km * np.cos(lat) * np.sin(lon)
    z = radius_km * np.sin(lat)
    return np.array([x, y, z])

def cartesian_to_latlon_xyz(xyz):
    x, y, z = xyz
    r = norm(xyz)
    if r == 0:
        return 0.0, 0.0
    lat = np.degrees(np.arcsin(z / r))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon

def gmst_deg_utc(dt):
    jd = jd_from_datetime(dt)
    T = (jd - 2451545.0) / 36525.0
    gmst = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - (T**3) / 38710000.0
    )
    return np.mod(gmst, 360.0)

def solar_vector_eci(dt):
    jd = jd_from_datetime(dt)
    n = jd - 2451545.0

    L = np.deg2rad((280.460 + 0.9856474 * n) % 360.0)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360.0)
    lam = L + np.deg2rad(1.915 * np.sin(g) + 0.020 * np.sin(2.0 * g))
    eps = np.deg2rad(23.439 - 0.0000004 * n)

    x = np.cos(lam)
    y = np.cos(eps) * np.sin(lam)
    z = np.sin(eps) * np.sin(lam)
    return np.array([x, y, z])

def local_enu_basis(lat_deg, lon_deg):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)

    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
        np.cos(lat)
    ])
    up = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])
    return east, north, up

def to_enu(vector, lat_deg, lon_deg):
    east, north, up = local_enu_basis(lat_deg, lon_deg)
    e = np.dot(east, vector)
    n = np.dot(north, vector)
    u = np.dot(up, vector)
    return np.array([e, n, u])

def azimuth_from_enu(enu_vec):
    e, n, u = enu_vec
    az = np.degrees(np.arctan2(e, n))
    if az < 0:
        az += 360.0
    return az

def elevation_from_enu(enu_vec):
    e, n, u = enu_vec
    r = norm(enu_vec)
    if r == 0:
        return 0.0
    return np.degrees(np.arcsin(u / r))

def local_solar_time_from_lon_utc(dt_utc, lon_deg):
    utc_hours = dt_utc.hour + dt_utc.minute / 60.0 + dt_utc.second / 3600.0
    lst_hours = utc_hours + lon_deg / 15.0
    return lst_hours % 24.0

def orbit_position_eci_from_t0(epoch_dt, t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg):
    a = R_EARTH + altitude_km
    n = np.sqrt(MU_EARTH / a**3)
    M0 = np.deg2rad(mean_anomaly_deg)
    M = M0 + n * t_seconds

    x_orb = a * np.cos(M)
    y_orb = a * np.sin(M)
    z_orb = 0.0
    pos_orb = np.array([x_orb, y_orb, z_orb])

    Rz_raan = rotation_z(np.deg2rad(raan_deg))
    Rx_i = rotation_x(np.deg2rad(inclination_deg))
    Rz_argp = rotation_z(np.deg2rad(argp_deg))
    R = Rz_raan @ Rx_i @ Rz_argp

    pos_eci = R @ pos_orb
    return pos_eci

def generate_orbit_track(
    orbit_name,
    altitude_km,
    inclination_deg,
    raan_deg,
    argp_deg,
    mean_anomaly_deg,
    start_utc,
    end_utc,
    step_minutes=10
):
    records = []
    dt = start_utc
    step_td = timedelta(minutes=step_minutes)

    while dt <= end_utc:
        gmst_deg = gmst_deg_utc(dt)
        gmst_rad = np.deg2rad(gmst_deg)

        t_seconds = (dt - start_utc).total_seconds()
        pos_eci = orbit_position_eci_from_t0(
            start_utc, t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg
        )

        # ECI -> ECEF
        pos_ecef = rotation_z(gmst_rad) @ pos_eci

        lat_deg, lon_deg = cartesian_to_latlon_xyz(pos_ecef)
        lon_deg = (lon_deg + 180.0) % 360.0 - 180.0

        obs_ecef = spherical_to_cartesian(lat_deg, lon_deg, R_EARTH)

        sat_rel = pos_ecef - obs_ecef
        sat_enu = to_enu(sat_rel, lat_deg, lon_deg)
        sat_az = azimuth_from_enu(sat_enu)
        sat_el = elevation_from_enu(sat_enu)

        sun_eci = solar_vector_eci(dt)
        sun_ecef = rotation_z(gmst_rad) @ sun_eci
        sun_rel = sun_ecef - obs_ecef
        sun_enu = to_enu(sun_rel, lat_deg, lon_deg)
        sun_az = azimuth_from_enu(sun_enu)
        sun_el = elevation_from_enu(sun_enu)
        sun_zenith = 90.0 - sun_el
        sunlit = sun_el > 0.0

        utc_iso = dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        local_solar_time_h = local_solar_time_from_lon_utc(dt, lon_deg)
        total_minutes = int(round(local_solar_time_h * 60.0))
        hh = total_minutes // 60
        mm = total_minutes % 60
        local_solar_time_hhmm = f"{hh:02d}:{mm:02d}"

        records.append({
            "orbit_name": orbit_name,
            "utc_time": utc_iso,
            "unix_seconds": int(dt.timestamp()),
            "latitude_deg": float(lat_deg),
            "longitude_deg": float(lon_deg),
            "satellite_azimuth_deg": float(sat_az),
            "satellite_elevation_deg": float(sat_el),
            "sun_azimuth_deg": float(sun_az),
            "sun_elevation_deg": float(sun_el),
            "sun_zenith_deg": float(sun_zenith),
            "local_solar_time_h": float(local_solar_time_h),
            "local_solar_time_hhmm": local_solar_time_hhmm,
            "sunlit": bool(sunlit),
            "altitude_km": float(altitude_km),
            "inclination_deg": float(inclination_deg),
            "raan_deg": float(raan_deg),
            "argp_deg": float(argp_deg),
            "mean_anomaly_deg": float(mean_anomaly_deg),
        })

        dt += step_td

    return pd.DataFrame(records)

# ============================================================
# Test case: 35-degree inclination orbit
# ============================================================

start_utc = datetime(2026, 1, 1, 0, 0, tzinfo=timezone.utc)
end_utc = start_utc + timedelta(days=1)

track_df = generate_orbit_track(
    orbit_name="inc35_test",
    altitude_km=700.0,
    inclination_deg=35.0,
    raan_deg=0.0,
    argp_deg=0.0,
    mean_anomaly_deg=0.0,
    start_utc=start_utc,
    end_utc=end_utc,
    step_minutes=10
)

# Write text output (CSV)
csv_path = "osse_inc35_orbit.csv"
track_df.to_csv(csv_path, index=False)
print(f"CSV written: {csv_path}")

# Write NetCDF
nc_path = "osse_inc35_orbit.nc"
with Dataset(nc_path, "w", format="NETCDF4") as nc:
    n = len(track_df)

    time_dim = nc.createDimension("time", n)
    lat_var = nc.createVariable("latitude_deg", "f8", ("time",))
    lon_var = nc.createVariable("longitude_deg", "f8", ("time",))
    sat_az_var = nc.createVariable("satellite_azimuth_deg", "f8", ("time",))
    sat_el_var = nc.createVariable("satellite_elevation_deg", "f8", ("time",))
    sun_az_var = nc.createVariable("sun_azimuth_deg", "f8", ("time",))
    sun_el_var = nc.createVariable("sun_elevation_deg", "f8", ("time",))
    sun_zen_var = nc.createVariable("sun_zenith_deg", "f8", ("time",))
    local_lst_var = nc.createVariable("local_solar_time_h", "f8", ("time",))
    sunlit_var = nc.createVariable("sunlit", "i1", ("time",))
    unix_sec_var = nc.createVariable("unix_seconds", "i8", ("time",))

    lat_var[:] = track_df["latitude_deg"].values
    lon_var[:] = track_df["longitude_deg"].values
    sat_az_var[:] = track_df["satellite_azimuth_deg"].values
    sat_el_var[:] = track_df["satellite_elevation_deg"].values
    sun_az_var[:] = track_df["sun_azimuth_deg"].values
    sun_el_var[:] = track_df["sun_elevation_deg"].values
    sun_zen_var[:] = track_df["sun_zenith_deg"].values
    local_lst_var[:] = track_df["local_solar_time_h"].values
    sunlit_var[:] = track_df["sunlit"].astype(np.int8).values
    unix_sec_var[:] = track_df["unix_seconds"].values

    nc.setncattr("orbit_name", "inc35_test")
    nc.setncattr("altitude_km", 700.0)
    nc.setncattr("inclination_deg", 35.0)
    nc.setncattr("raan_deg", 0.0)
    nc.setncattr("argp_deg", 0.0)
    nc.setncattr("mean_anomaly_deg", 0.0)
    nc.setncattr("start_utc", start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("end_utc", end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("step_minutes", 10)

print(f"NetCDF written: {nc_path}")

# ============================================================
# Orbit track plots
# ============================================================

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
ax = axs[0]
sc = ax.scatter(track_df["longitude_deg"], track_df["latitude_deg"], c=track_df["sun_elevation_deg"], cmap="viridis", s=10, edgecolor="none")
ax.set_title("35° Inclination Orbit Track (Lon vs Lat)")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.grid(True, alpha=0.3)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Sun elevation [deg]")

ax = axs[1]
ax.plot(track_df["longitude_deg"], track_df["latitude_deg"], color="tab:blue", lw=1.5)
ax.set_title("Orbit Track")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = "osse_inc35_orbit_track.png"
plt.savefig(plot_path, dpi=200)
print(f"Plot written: {plot_path}")

print("\nSample rows:")
print(track_df.head().to_string(index=False))

print("\nOutputs created:")
print(" -", csv_path)
print(" -", nc_path)
print(" -", plot_path)

CSV written: osse_inc35_orbit.csv
NetCDF written: osse_inc35_orbit.nc
Plot written: osse_inc35_orbit_track.png

Sample rows:
orbit_name             utc_time  unix_seconds  latitude_deg  longitude_deg  satellite_azimuth_deg  satellite_elevation_deg  sun_azimuth_deg  sun_elevation_deg  sun_zenith_deg  local_solar_time_h local_solar_time_hhmm  sunlit  altitude_km  inclination_deg  raan_deg  argp_deg  mean_anomaly_deg
inc35_test 2026-01-01T00:00:00Z    1767225600      0.000000     100.660859              90.000000                     90.0       246.564799         -89.991169      179.991169            6.710724                 06:43   False        700.0             35.0       0.0       0.0               0.0
inc35_test 2026-01-01T00:10:00Z    1767226200     19.922660     134.340530              90.000000                     90.0       253.181036         -89.991867      179.991867            9.122702                 09:07   False        700.0             35.0       0.0       0.0               

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from netCDF4 import Dataset

# ============================================================
# Orbit generation with variable footprint resolution:
# - 15 km at nadir
# - slightly elongated near swath edges
# - outputs CSV + NetCDF + orbit plot
# ============================================================

R_EARTH = 6378.137
MU_EARTH = 398600.4418

def deg2rad(x):
    return np.deg2rad(x)

def norm(v):
    return np.linalg.norm(v)

def rotation_z(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

def rotation_x(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0,  c, -s],
        [0.0,  s,  c]
    ])

def spherical_to_cartesian(lat_deg, lon_deg, radius_km=R_EARTH):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    x = radius_km * np.cos(lat) * np.cos(lon)
    y = radius_km * np.cos(lat) * np.sin(lon)
    z = radius_km * np.sin(lat)
    return np.array([x, y, z])

def cartesian_to_latlon_xyz(xyz):
    x, y, z = xyz
    r = norm(xyz)
    if r == 0:
        return 0.0, 0.0
    lat = np.degrees(np.arcsin(z / r))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon

def jd_from_datetime(dt):
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    return 2440587.5 + (dt - epoch).total_seconds() / 86400.0

def gmst_deg_utc(dt):
    jd = jd_from_datetime(dt)
    T = (jd - 2451545.0) / 36525.0
    gmst = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - (T**3) / 38710000.0
    )
    return np.mod(gmst, 360.0)

def solar_vector_eci(dt):
    jd = jd_from_datetime(dt)
    n = jd - 2451545.0
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360.0)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360.0)
    lam = L + np.deg2rad(1.915*np.sin(g) + 0.020*np.sin(2.0*g))
    eps = np.deg2rad(23.439 - 0.0000004*n)
    x = np.cos(lam)
    y = np.cos(eps) * np.sin(lam)
    z = np.sin(eps) * np.sin(lam)
    return np.array([x, y, z])

def local_enu_basis(lat_deg, lon_deg):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
        np.cos(lat)
    ])
    up = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])
    return east, north, up

def to_enu(vector, lat_deg, lon_deg):
    east, north, up = local_enu_basis(lat_deg, lon_deg)
    e = np.dot(east, vector)
    n = np.dot(north, vector)
    u = np.dot(up, vector)
    return np.array([e, n, u])

def azimuth_from_enu(enu_vec):
    e, n, u = enu_vec
    az = np.degrees(np.arctan2(e, n))
    if az < 0:
        az += 360.0
    return az

def elevation_from_enu(enu_vec):
    e, n, u = enu_vec
    r = norm(enu_vec)
    if r == 0:
        return 0.0
    return np.degrees(np.arcsin(u / r))

def local_solar_time_from_lon_utc(dt_utc, lon_deg):
    utc_hours = dt_utc.hour + dt_utc.minute / 60.0 + dt_utc.second / 3600.0
    lst_hours = utc_hours + lon_deg / 15.0
    return lst_hours % 24.0

def orbit_position_eci_from_t0(t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg):
    a = R_EARTH + altitude_km
    n = np.sqrt(MU_EARTH / a**3)
    M0 = np.deg2rad(mean_anomaly_deg)
    M = M0 + n * t_seconds

    x_orb = a * np.cos(M)
    y_orb = a * np.sin(M)
    z_orb = 0.0
    pos_orb = np.array([x_orb, y_orb, z_orb])

    Rz_raan = rotation_z(np.deg2rad(raan_deg))
    Rx_i = rotation_x(np.deg2rad(inclination_deg))
    Rz_argp = rotation_z(np.deg2rad(argp_deg))
    R = Rz_raan @ Rx_i @ Rz_argp

    return R @ pos_orb

def footprint_resolution_km(off_nadir_deg, nadir_km=15.0, edge_deg=40.0, elongation=0.12):
    theta = min(abs(off_nadir_deg), edge_deg)
    theta_rad = np.deg2rad(theta)
    cross_track = nadir_km / np.cos(theta_rad)
    along_track = cross_track * (1.0 + elongation * (theta / edge_deg))
    return cross_track, along_track

def generate_orbit_track(
    orbit_name,
    altitude_km,
    inclination_deg,
    raan_deg,
    argp_deg,
    mean_anomaly_deg,
    start_utc,
    end_utc,
    step_minutes=10,
    max_off_nadir_deg=40.0,
    nadir_resolution_km=15.0
):
    records = []
    dt = start_utc
    step_td = timedelta(minutes=step_minutes)

    while dt <= end_utc:
        gmst_deg = gmst_deg_utc(dt)
        gmst_rad = np.deg2rad(gmst_deg)

        t_seconds = (dt - start_utc).total_seconds()
        pos_eci = orbit_position_eci_from_t0(
            t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg
        )
        pos_ecef = rotation_z(gmst_rad) @ pos_eci

        lat_deg, lon_deg = cartesian_to_latlon_xyz(pos_ecef)
        lon_deg = (lon_deg + 180.0) % 360.0 - 180.0

        obs_ecef = spherical_to_cartesian(lat_deg, lon_deg, R_EARTH)
        sat_rel = pos_ecef - obs_ecef
        sat_enu = to_enu(sat_rel, lat_deg, lon_deg)
        sat_az = azimuth_from_enu(sat_enu)
        sat_el = elevation_from_enu(sat_enu)

        # approximate off-nadir angle based on satellite view from sub-satellite point
        # using zenith angle relative to local vertical
        off_nadir = 90.0 - sat_el
        off_nadir = max(0.0, min(off_nadir, max_off_nadir_deg))

        cross_res, along_res = footprint_resolution_km(
            off_nadir_deg=off_nadir,
            nadir_km=nadir_resolution_km,
            edge_deg=max_off_nadir_deg,
            elongation=0.12
        )

        # sun geometry
        sun_eci = solar_vector_eci(dt)
        sun_ecef = rotation_z(gmst_rad) @ sun_eci
        sun_rel = sun_ecef - obs_ecef
        sun_enu = to_enu(sun_rel, lat_deg, lon_deg)
        sun_az = azimuth_from_enu(sun_enu)
        sun_el = elevation_from_enu(sun_enu)
        sun_zenith = 90.0 - sun_el
        sunlit = sun_el > 0.0

        local_solar_time_h = local_solar_time_from_lon_utc(dt, lon_deg)
        total_minutes = int(round(local_solar_time_h * 60.0))
        hh = total_minutes // 60
        mm = total_minutes % 60
        local_solar_time_hhmm = f"{hh:02d}:{mm:02d}"

        records.append({
            "orbit_name": orbit_name,
            "utc_time": dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "unix_seconds": int(dt.timestamp()),
            "latitude_deg": float(lat_deg),
            "longitude_deg": float(lon_deg),
            "off_nadir_angle_deg": float(off_nadir),
            "cross_track_resolution_km": float(cross_res),
            "along_track_resolution_km": float(along_res),
            "pixel_area_km2": float(cross_res * along_res),
            "swath_edge_flag": bool(off_nadir > 0.8 * max_off_nadir_deg),
            "satellite_azimuth_deg": float(sat_az),
            "satellite_elevation_deg": float(sat_el),
            "sun_azimuth_deg": float(sun_az),
            "sun_elevation_deg": float(sun_el),
            "sun_zenith_deg": float(sun_zenith),
            "local_solar_time_h": float(local_solar_time_h),
            "local_solar_time_hhmm": local_solar_time_hhmm,
            "sunlit": bool(sunlit),
            "altitude_km": float(altitude_km),
            "inclination_deg": float(inclination_deg),
            "raan_deg": float(raan_deg),
            "argp_deg": float(argp_deg),
            "mean_anomaly_deg": float(mean_anomaly_deg),
        })

        dt += step_td

    return pd.DataFrame(records)

# ============================================================
# Example: 35 degree inclination orbit with swath resolution
# ============================================================

start_utc = datetime(2026, 1, 1, 0, 0, tzinfo=timezone.utc)
end_utc = start_utc + timedelta(days=1)

track_df = generate_orbit_track(
    orbit_name="inc35_variable_resolution",
    altitude_km=700.0,
    inclination_deg=35.0,
    raan_deg=0.0,
    argp_deg=0.0,
    mean_anomaly_deg=0.0,
    start_utc=start_utc,
    end_utc=end_utc,
    step_minutes=10,
    max_off_nadir_deg=40.0,
    nadir_resolution_km=15.0
)

# Save CSV
csv_path = "osse_inc35_variable_resolution.csv"
track_df.to_csv(csv_path, index=False)
print(f"CSV written: {csv_path}")

# Save NetCDF
nc_path = "osse_inc35_variable_resolution.nc"
with Dataset(nc_path, "w", format="NETCDF4") as nc:
    n = len(track_df)

    time_dim = nc.createDimension("time", n)
    lat_var = nc.createVariable("latitude_deg", "f8", ("time",))
    lon_var = nc.createVariable("longitude_deg", "f8", ("time",))
    off_nadir_var = nc.createVariable("off_nadir_angle_deg", "f8", ("time",))
    cross_var = nc.createVariable("cross_track_resolution_km", "f8", ("time",))
    along_var = nc.createVariable("along_track_resolution_km", "f8", ("time",))
    area_var = nc.createVariable("pixel_area_km2", "f8", ("time",))
    swath_var = nc.createVariable("swath_edge_flag", "i1", ("time",))
    sun_el_var = nc.createVariable("sun_elevation_deg", "f8", ("time",))

    lat_var[:] = track_df["latitude_deg"].values
    lon_var[:] = track_df["longitude_deg"].values
    off_nadir_var[:] = track_df["off_nadir_angle_deg"].values
    cross_var[:] = track_df["cross_track_resolution_km"].values
    along_var[:] = track_df["along_track_resolution_km"].values
    area_var[:] = track_df["pixel_area_km2"].values
    swath_var[:] = track_df["swath_edge_flag"].astype(np.int8).values
    sun_el_var[:] = track_df["sun_elevation_deg"].values

    nc.setncattr("orbit_name", "inc35_variable_resolution")
    nc.setncattr("nadir_resolution_km", 15.0)
    nc.setncattr("max_off_nadir_deg", 40.0)
    nc.setncattr("inclination_deg", 35.0)
    nc.setncattr("start_utc", start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("end_utc", end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("step_minutes", 10)

print(f"NetCDF written: {nc_path}")

# Plot orbit track
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

ax = axs[0]
sc = ax.scatter(track_df["longitude_deg"], track_df["latitude_deg"], c=track_df["cross_track_resolution_km"], cmap="plasma", s=10, edgecolor="none")
ax.set_title("Orbit Track with Variable Resolution")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.grid(True, alpha=0.3)
fig.colorbar(sc, ax=ax, label="Cross-track resolution [km]")

ax = axs[1]
ax.plot(track_df["longitude_deg"], track_df["latitude_deg"], color="tab:blue", lw=1.5)
ax.scatter(
    track_df.loc[track_df["swath_edge_flag"], "longitude_deg"],
    track_df.loc[track_df["swath_edge_flag"], "latitude_deg"],
    color="red", s=8, label="Swath edge"
)
ax.set_title("Orbit Track and Swath Edge Points")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plot_path = "osse_inc35_variable_resolution_track.png"
plt.savefig(plot_path, dpi=200)
print(f"Plot written: {plot_path}")

print(track_df.head().to_string(index=False))
print("\nComplete orbit data saved with variable spatial resolution.")

CSV written: osse_inc35_variable_resolution.csv
NetCDF written: osse_inc35_variable_resolution.nc
Plot written: osse_inc35_variable_resolution_track.png
               orbit_name             utc_time  unix_seconds  latitude_deg  longitude_deg  off_nadir_angle_deg  cross_track_resolution_km  along_track_resolution_km  pixel_area_km2  swath_edge_flag  satellite_azimuth_deg  satellite_elevation_deg  sun_azimuth_deg  sun_elevation_deg  sun_zenith_deg  local_solar_time_h local_solar_time_hhmm  sunlit  altitude_km  inclination_deg  raan_deg  argp_deg  mean_anomaly_deg
inc35_variable_resolution 2026-01-01T00:00:00Z    1767225600      0.000000     100.660859                  0.0                       15.0                       15.0           225.0            False              90.000000                     90.0       246.564799         -89.991169      179.991169            6.710724                 06:43   False        700.0             35.0       0.0       0.0               0.0
inc35_variable_

In [4]:
!pip install netCDF4 cartopy shapely pyproj

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 23.2 MB/s  0:00:00


In [5]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from netCDF4 import Dataset

# ============================================================
# Orbit track generation with 15 km sampling along track
# and Earth map plotting
# ============================================================

R_EARTH = 6378.137
MU_EARTH = 398600.4418

def deg2rad(x):
    return np.deg2rad(x)

def norm(v):
    return np.linalg.norm(v)

def rotation_z(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

def rotation_x(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0,  c, -s],
        [0.0,  s,  c]
    ])

def spherical_to_cartesian(lat_deg, lon_deg, radius_km=R_EARTH):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    x = radius_km * np.cos(lat) * np.cos(lon)
    y = radius_km * np.cos(lat) * np.sin(lon)
    z = radius_km * np.sin(lat)
    return np.array([x, y, z])

def cartesian_to_latlon_xyz(xyz):
    x, y, z = xyz
    r = norm(xyz)
    if r == 0:
        return 0.0, 0.0
    lat = np.degrees(np.arcsin(z / r))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon

def jd_from_datetime(dt):
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    return 2440587.5 + (dt - epoch).total_seconds() / 86400.0

def gmst_deg_utc(dt):
    jd = jd_from_datetime(dt)
    T = (jd - 2451545.0) / 36525.0
    gmst = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - (T**3) / 38710000.0
    )
    return np.mod(gmst, 360.0)

def solar_vector_eci(dt):
    jd = jd_from_datetime(dt)
    n = jd - 2451545.0
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360.0)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360.0)
    lam = L + np.deg2rad(1.915 * np.sin(g) + 0.020 * np.sin(2.0 * g))
    eps = np.deg2rad(23.439 - 0.0000004 * n)

    x = np.cos(lam)
    y = np.cos(eps) * np.sin(lam)
    z = np.sin(eps) * np.sin(lam)
    return np.array([x, y, z])

def local_enu_basis(lat_deg, lon_deg):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)

    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
        np.cos(lat)
    ])
    up = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])
    return east, north, up

def to_enu(vector, lat_deg, lon_deg):
    east, north, up = local_enu_basis(lat_deg, lon_deg)
    e = np.dot(east, vector)
    n = np.dot(north, vector)
    u = np.dot(up, vector)
    return np.array([e, n, u])

def azimuth_from_enu(enu_vec):
    e, n, u = enu_vec
    az = np.degrees(np.arctan2(e, n))
    if az < 0:
        az += 360.0
    return az

def elevation_from_enu(enu_vec):
    e, n, u = enu_vec
    r = norm(enu_vec)
    if r == 0:
        return 0.0
    return np.degrees(np.arcsin(u / r))

def local_solar_time_from_lon_utc(dt_utc, lon_deg):
    utc_hours = dt_utc.hour + dt_utc.minute / 60.0 + dt_utc.second / 3600.0
    lst_hours = utc_hours + lon_deg / 15.0
    return lst_hours % 24.0

def orbit_position_eci_from_t0(t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg):
    a = R_EARTH + altitude_km
    n = np.sqrt(MU_EARTH / a**3)
    M0 = np.deg2rad(mean_anomaly_deg)
    M = M0 + n * t_seconds

    x_orb = a * np.cos(M)
    y_orb = a * np.sin(M)
    z_orb = 0.0
    pos_orb = np.array([x_orb, y_orb, z_orb])

    Rz_raan = rotation_z(np.deg2rad(raan_deg))
    Rx_i = rotation_x(np.deg2rad(inclination_deg))
    Rz_argp = rotation_z(np.deg2rad(argp_deg))
    R = Rz_raan @ Rx_i @ Rz_argp
    return R @ pos_orb

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

def footprint_resolution_km(off_nadir_deg, nadir_km=15.0, edge_deg=40.0, elongation=0.12):
    theta = min(abs(off_nadir_deg), edge_deg)
    theta_rad = np.deg2rad(theta)
    cross_track = nadir_km / np.cos(theta_rad)
    along_track = cross_track * (1.0 + elongation * (theta / edge_deg))
    return cross_track, along_track

def generate_fine_orbit_track(
    orbit_name,
    altitude_km,
    inclination_deg,
    raan_deg,
    argp_deg,
    mean_anomaly_deg,
    start_utc,
    end_utc,
    step_minutes=2,
    max_off_nadir_deg=40.0,
    nadir_resolution_km=15.0
):
    records = []
    dt = start_utc
    step_td = timedelta(minutes=step_minutes)

    while dt <= end_utc:
        gmst_deg = gmst_deg_utc(dt)
        gmst_rad = np.deg2rad(gmst_deg)

        t_seconds = (dt - start_utc).total_seconds()
        pos_eci = orbit_position_eci_from_t0(
            t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg
        )
        pos_ecef = rotation_z(gmst_rad) @ pos_eci

        lat_deg, lon_deg = cartesian_to_latlon_xyz(pos_ecef)
        lon_deg = (lon_deg + 180.0) % 360.0 - 180.0

        obs_ecef = spherical_to_cartesian(lat_deg, lon_deg, R_EARTH)
        sat_rel = pos_ecef - obs_ecef
        sat_enu = to_enu(sat_rel, lat_deg, lon_deg)
        sat_az = azimuth_from_enu(sat_enu)
        sat_el = elevation_from_enu(sat_enu)

        off_nadir = 90.0 - sat_el
        off_nadir = max(0.0, min(off_nadir, max_off_nadir_deg))

        cross_res, along_res = footprint_resolution_km(
            off_nadir_deg=off_nadir,
            nadir_km=nadir_resolution_km,
            edge_deg=max_off_nadir_deg,
            elongation=0.12
        )

        sun_eci = solar_vector_eci(dt)
        sun_ecef = rotation_z(gmst_rad) @ sun_eci
        sun_rel = sun_ecef - obs_ecef
        sun_enu = to_enu(sun_rel, lat_deg, lon_deg)
        sun_az = azimuth_from_enu(sun_enu)
        sun_el = elevation_from_enu(sun_enu)
        sun_zenith = 90.0 - sun_el
        sunlit = sun_el > 0.0

        local_solar_time_h = local_solar_time_from_lon_utc(dt, lon_deg)
        total_minutes = int(round(local_solar_time_h * 60.0))
        hh = total_minutes // 60
        mm = total_minutes % 60
        lst_hhmm = f"{hh:02d}:{mm:02d}"

        records.append({
            "orbit_name": orbit_name,
            "utc_time": dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "unix_seconds": int(dt.timestamp()),
            "latitude_deg": float(lat_deg),
            "longitude_deg": float(lon_deg),
            "off_nadir_angle_deg": float(off_nadir),
            "cross_track_resolution_km": float(cross_res),
            "along_track_resolution_km": float(along_res),
            "pixel_area_km2": float(cross_res * along_res),
            "swath_edge_flag": bool(off_nadir > 0.8 * max_off_nadir_deg),
            "satellite_azimuth_deg": float(sat_az),
            "satellite_elevation_deg": float(sat_el),
            "sun_azimuth_deg": float(sun_az),
            "sun_elevation_deg": float(sun_el),
            "sun_zenith_deg": float(sun_zenith),
            "local_solar_time_h": float(local_solar_time_h),
            "local_solar_time_hhmm": lst_hhmm,
            "sunlit": bool(sunlit),
            "altitude_km": float(altitude_km),
            "inclination_deg": float(inclination_deg),
            "raan_deg": float(raan_deg),
            "argp_deg": float(argp_deg),
            "mean_anomaly_deg": float(mean_anomaly_deg),
        })

        dt += step_td

    return pd.DataFrame(records)

def sample_track_every_15km(df, interval_km=15.0):
    # df must have latitude_deg and longitude_deg in order along orbit
    if len(df) < 2:
        return df.copy()

    # cumulative distance along track
    cum = np.zeros(len(df))
    for i in range(1, len(df)):
        cum[i] = cum[i-1] + haversine_km(
            df["latitude_deg"].iloc[i-1],
            df["longitude_deg"].iloc[i-1],
            df["latitude_deg"].iloc[i],
            df["longitude_deg"].iloc[i]
        )

    total_dist = cum[-1]
    sample_dist = np.arange(0.0, total_dist + interval_km, interval_km)

    out = {}
    for col in df.columns:
        if col in ["latitude_deg", "longitude_deg", "off_nadir_angle_deg",
                   "cross_track_resolution_km", "along_track_resolution_km",
                   "pixel_area_km2", "satellite_azimuth_deg", "satellite_elevation_deg",
                   "sun_azimuth_deg", "sun_elevation_deg", "sun_zenith_deg",
                   "local_solar_time_h", "sunlit", "altitude_km", "inclination_deg",
                   "raan_deg", "argp_deg", "mean_anomaly_deg", "unix_seconds"]:
            out[col] = np.interp(sample_dist, cum, df[col].values)

    # For string/time columns, use nearest neighbor
    if "utc_time" in df.columns:
        out["utc_time"] = [df["utc_time"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist]
    if "local_solar_time_hhmm" in df.columns:
        out["local_solar_time_hhmm"] = [df["local_solar_time_hhmm"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist]
    if "orbit_name" in df.columns:
        out["orbit_name"] = [df["orbit_name"].iloc[0]] * len(sample_dist)

    out_df = pd.DataFrame(out)

    # ensure boolean and numeric columns exist
    if "sunlit" in out_df.columns:
        out_df["sunlit"] = out_df["sunlit"].astype(bool)

    if "swath_edge_flag" in df.columns:
        out_df["swath_edge_flag"] = np.interp(sample_dist, cum, df["swath_edge_flag"].astype(float).values).astype(bool)

    out_df["distance_km"] = sample_dist

    return out_df

# ============================================================
# Main generation
# ============================================================

start_utc = datetime(2026, 1, 1, 0, 0, tzinfo=timezone.utc)
end_utc = start_utc + timedelta(days=1)

fine_df = generate_fine_orbit_track(
    orbit_name="inc35_test",
    altitude_km=700.0,
    inclination_deg=35.0,
    raan_deg=0.0,
    argp_deg=0.0,
    mean_anomaly_deg=0.0,
    start_utc=start_utc,
    end_utc=end_utc,
    step_minutes=2,
    max_off_nadir_deg=40.0,
    nadir_resolution_km=15.0
)

# Sample every 15 km along track
sampled_df = sample_track_every_15km(fine_df, interval_km=15.0)

# Save CSV
csv_path = "osse_inc35_track_15km.csv"
sampled_df.to_csv(csv_path, index=False)
print(f"CSV written: {csv_path}")

# Save NetCDF
nc_path = "osse_inc35_track_15km.nc"
with Dataset(nc_path, "w", format="NETCDF4") as nc:
    n = len(sampled_df)

    time_dim = nc.createDimension("time", n)
    lat_var = nc.createVariable("latitude_deg", "f8", ("time",))
    lon_var = nc.createVariable("longitude_deg", "f8", ("time",))
    dist_var = nc.createVariable("distance_km", "f8", ("time",))
    off_nadir_var = nc.createVariable("off_nadir_angle_deg", "f8", ("time",))
    cross_var = nc.createVariable("cross_track_resolution_km", "f8", ("time",))
    along_var = nc.createVariable("along_track_resolution_km", "f8", ("time",))
    area_var = nc.createVariable("pixel_area_km2", "f8", ("time",))
    sun_el_var = nc.createVariable("sun_elevation_deg", "f8", ("time",))
    sun_zen_var = nc.createVariable("sun_zenith_deg", "f8", ("time",))
    swath_var = nc.createVariable("swath_edge_flag", "i1", ("time",))
    sunlit_var = nc.createVariable("sunlit", "i1", ("time",))
    unix_sec_var = nc.createVariable("unix_seconds", "i8", ("time",))

    lat_var[:] = sampled_df["latitude_deg"].values
    lon_var[:] = sampled_df["longitude_deg"].values
    dist_var[:] = sampled_df["distance_km"].values
    off_nadir_var[:] = sampled_df["off_nadir_angle_deg"].values
    cross_var[:] = sampled_df["cross_track_resolution_km"].values
    along_var[:] = sampled_df["along_track_resolution_km"].values
    area_var[:] = sampled_df["pixel_area_km2"].values
    sun_el_var[:] = sampled_df["sun_elevation_deg"].values
    sun_zen_var[:] = sampled_df["sun_zenith_deg"].values
    swath_var[:] = sampled_df["swath_edge_flag"].astype(np.int8).values
    sunlit_var[:] = sampled_df["sunlit"].astype(np.int8).values
    unix_sec_var[:] = sampled_df["unix_seconds"].values

    nc.setncattr("orbit_name", "inc35_test")
    nc.setncattr("interval_km", 15.0)
    nc.setncattr("inclination_deg", 35.0)
    nc.setncattr("altitude_km", 700.0)
    nc.setncattr("start_utc", start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("end_utc", end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))

print(f"NetCDF written: {nc_path}")

# ============================================================
# Plot on an Earth map
# ============================================================

fig = plt.figure(figsize=(13, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#d9d9d9", edgecolor="black", linewidth=0.4)
ax.add_feature(cfeature.OCEAN, facecolor="#cfe8ff", edgecolor="none")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.2)
ax.add_feature(cfeature.LAKES, facecolor="#cfe8ff", edgecolor="none")

ax.plot(
    sampled_df["longitude_deg"],
    sampled_df["latitude_deg"],
    transform=ccrs.PlateCarree(),
    color="tab:red",
    linewidth=1.8,
    label="15 km orbit samples"
)

ax.set_title("35° Inclination Orbit Track on Earth Map")
ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)

plot_path = "osse_inc35_orbit_map_15km.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
print(f"Map plot written: {plot_path}")

print("\nSample rows:")
print(sampled_df.head().to_string(index=False))

print(f"\nGenerated {len(sampled_df)} samples at 15 km intervals.")

CSV written: osse_inc35_track_15km.csv
NetCDF written: osse_inc35_track_15km.nc


/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_ocean.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWar

Map plot written: osse_inc35_orbit_map_15km.png

Sample rows:
 unix_seconds  latitude_deg  longitude_deg  off_nadir_angle_deg  cross_track_resolution_km  along_track_resolution_km  pixel_area_km2  satellite_azimuth_deg  satellite_elevation_deg  sun_azimuth_deg  sun_elevation_deg  sun_zenith_deg  local_solar_time_h  sunlit  altitude_km  inclination_deg  raan_deg  argp_deg  mean_anomaly_deg             utc_time local_solar_time_hhmm orbit_name  swath_edge_flag  distance_km
 1.767226e+09      0.000000     100.660859                  0.0                       15.0                       15.0           225.0              90.000000                     90.0       246.564799         -89.991169      179.991169            6.710724   False        700.0             35.0       0.0       0.0               0.0 2026-01-01T00:00:00Z                 06:43 inc35_test            False          0.0
 1.767226e+09      0.073064     100.774357                  0.0                       15.0                    

In [6]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from netCDF4 import Dataset

# ============================================================
# OSSE orbit track with 15 km along-track spacing and 2200 km swath
# Outputs:
#   - osse_inc35_track_15km_swath.csv
#   - osse_inc35_track_15km_swath.nc
#   - osse_inc35_orbit_swath_map.png
# ============================================================

R_EARTH = 6378.137
MU_EARTH = 398600.4418

def deg2rad(x): return np.deg2rad(x)
def norm(v): return np.linalg.norm(v)

def rotation_z(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([[c, -s, 0.0],
                     [s,  c, 0.0],
                     [0.0, 0.0, 1.0]])

def rotation_x(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([[1.0, 0.0, 0.0],
                     [0.0,  c, -s],
                     [0.0,  s,  c]])

def spherical_to_cartesian(lat_deg, lon_deg, radius_km=R_EARTH):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    x = radius_km * np.cos(lat) * np.cos(lon)
    y = radius_km * np.cos(lat) * np.sin(lon)
    z = radius_km * np.sin(lat)
    return np.array([x, y, z])

def cartesian_to_latlon_xyz(xyz):
    x, y, z = xyz
    r = norm(xyz)
    if r == 0:
        return 0.0, 0.0
    lat = np.degrees(np.arcsin(z / r))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon

def jd_from_datetime(dt):
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    return 2440587.5 + (dt - epoch).total_seconds() / 86400.0

def gmst_deg_utc(dt):
    jd = jd_from_datetime(dt)
    T = (jd - 2451545.0) / 36525.0
    gmst = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - (T**3) / 38710000.0
    )
    return np.mod(gmst, 360.0)

def solar_vector_eci(dt):
    jd = jd_from_datetime(dt)
    n = jd - 2451545.0
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360.0)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360.0)
    lam = L + np.deg2rad(1.915 * np.sin(g) + 0.020 * np.sin(2.0 * g))
    eps = np.deg2rad(23.439 - 0.0000004 * n)

    x = np.cos(lam)
    y = np.cos(eps) * np.sin(lam)
    z = np.sin(eps) * np.sin(lam)
    return np.array([x, y, z])

def local_enu_basis(lat_deg, lon_deg):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([-np.sin(lat) * np.cos(lon),
                      -np.sin(lat) * np.sin(lon),
                      np.cos(lat)])
    up = np.array([np.cos(lat) * np.cos(lon),
                   np.cos(lat) * np.sin(lon),
                   np.sin(lat)])
    return east, north, up

def to_enu(vector, lat_deg, lon_deg):
    east, north, up = local_enu_basis(lat_deg, lon_deg)
    e = np.dot(east, vector)
    n = np.dot(north, vector)
    u = np.dot(up, vector)
    return np.array([e, n, u])

def azimuth_from_enu(enu_vec):
    e, n, u = enu_vec
    az = np.degrees(np.arctan2(e, n))
    if az < 0:
        az += 360.0
    return az

def elevation_from_enu(enu_vec):
    e, n, u = enu_vec
    r = norm(enu_vec)
    if r == 0:
        return 0.0
    return np.degrees(np.arcsin(u / r))

def local_solar_time_from_lon_utc(dt_utc, lon_deg):
    utc_hours = dt_utc.hour + dt_utc.minute / 60.0 + dt_utc.second / 3600.0
    lst_hours = utc_hours + lon_deg / 15.0
    return lst_hours % 24.0

def orbit_position_eci_from_t0(t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg):
    a = R_EARTH + altitude_km
    n = np.sqrt(MU_EARTH / a**3)
    M0 = np.deg2rad(mean_anomaly_deg)
    M = M0 + n * t_seconds

    x_orb = a * np.cos(M)
    y_orb = a * np.sin(M)
    z_orb = 0.0
    pos_orb = np.array([x_orb, y_orb, z_orb])

    Rz_raan = rotation_z(np.deg2rad(raan_deg))
    Rx_i = rotation_x(np.deg2rad(inclination_deg))
    Rz_argp = rotation_z(np.deg2rad(argp_deg))
    R = Rz_raan @ Rx_i @ Rz_argp
    return R @ pos_orb

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

def generate_fine_orbit_track(
    orbit_name, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg,
    start_utc, end_utc, step_minutes=2
):
    records = []
    dt = start_utc
    step_td = timedelta(minutes=step_minutes)

    while dt <= end_utc:
        gmst_deg = gmst_deg_utc(dt)
        gmst_rad = np.deg2rad(gmst_deg)

        t_seconds = (dt - start_utc).total_seconds()
        pos_eci = orbit_position_eci_from_t0(
            t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg
        )
        pos_ecef = rotation_z(gmst_rad) @ pos_eci

        lat_deg, lon_deg = cartesian_to_latlon_xyz(pos_ecef)
        lon_deg = (lon_deg + 180.0) % 360.0 - 180.0

        obs_ecef = spherical_to_cartesian(lat_deg, lon_deg, R_EARTH)
        sat_rel = pos_ecef - obs_ecef
        sat_enu = to_enu(sat_rel, lat_deg, lon_deg)
        sat_az = azimuth_from_enu(sat_enu)
        sat_el = elevation_from_enu(sat_enu)
        off_nadir = 90.0 - sat_el

        sun_eci = solar_vector_eci(dt)
        sun_ecef = rotation_z(gmst_rad) @ sun_eci
        sun_rel = sun_ecef - obs_ecef
        sun_enu = to_enu(sun_rel, lat_deg, lon_deg)
        sun_az = azimuth_from_enu(sun_enu)
        sun_el = elevation_from_enu(sun_enu)
        sun_zenith = 90.0 - sun_el
        sunlit = sun_el > 0.0

        local_solar_time_h = local_solar_time_from_lon_utc(dt, lon_deg)
        total_minutes = int(round(local_solar_time_h * 60.0))
        hh = total_minutes // 60
        mm = total_minutes % 60
        local_solar_time_hhmm = f"{hh:02d}:{mm:02d}"

        records.append({
            "orbit_name": orbit_name,
            "utc_time": dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "unix_seconds": int(dt.timestamp()),
            "latitude_deg": float(lat_deg),
            "longitude_deg": float(lon_deg),
            "satellite_azimuth_deg": float(sat_az),
            "satellite_elevation_deg": float(sat_el),
            "sun_azimuth_deg": float(sun_az),
            "sun_elevation_deg": float(sun_el),
            "sun_zenith_deg": float(sun_zenith),
            "local_solar_time_h": float(local_solar_time_h),
            "local_solar_time_hhmm": local_solar_time_hhmm,
            "sunlit": bool(sunlit),
            "off_nadir_angle_deg": float(off_nadir),
            "altitude_km": float(altitude_km),
            "inclination_deg": float(inclination_deg),
            "raan_deg": float(raan_deg),
            "argp_deg": float(argp_deg),
            "mean_anomaly_deg": float(mean_anomaly_deg),
        })

        dt += step_td

    return pd.DataFrame(records)

def resample_track_every_15km(df, interval_km=15.0):
    if len(df) < 2:
        return df.copy()

    cum = np.zeros(len(df))
    for i in range(1, len(df)):
        cum[i] = cum[i-1] + haversine_km(
            df["latitude_deg"].iloc[i-1],
            df["longitude_deg"].iloc[i-1],
            df["latitude_deg"].iloc[i],
            df["longitude_deg"].iloc[i]
        )

    total_dist = cum[-1]
    sample_dist = np.arange(0.0, total_dist + interval_km, interval_km)

    out = {}
    for col in df.columns:
        if col in ["latitude_deg", "longitude_deg", "satellite_azimuth_deg",
                   "satellite_elevation_deg", "sun_azimuth_deg", "sun_elevation_deg",
                   "sun_zenith_deg", "local_solar_time_h", "off_nadir_angle_deg",
                   "altitude_km", "inclination_deg", "raan_deg", "argp_deg",
                   "mean_anomaly_deg", "unix_seconds"]:
            out[col] = np.interp(sample_dist, cum, df[col].values)

    if "utc_time" in df.columns:
        out["utc_time"] = [df["utc_time"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist]
    if "local_solar_time_hhmm" in df.columns:
        out["local_solar_time_hhmm"] = [df["local_solar_time_hhmm"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist]
    if "sunlit" in df.columns:
        out["sunlit"] = np.array([df["sunlit"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist], dtype=bool)
    if "orbit_name" in df.columns:
        out["orbit_name"] = [df["orbit_name"].iloc[0]] * len(sample_dist)

    out["distance_km"] = sample_dist
    return pd.DataFrame(out)

def make_swath_at_track(track_df, swath_width_km=2200.0, step_km=50.0):
    """
    Create swath points around each track point.
    For each orbit point, generate cross-track positions from -1100 to +1100 km.
    """
    rows = []

    for idx, row in track_df.iterrows():
        lat0 = float(row["latitude_deg"])
        lon0 = float(row["longitude_deg"])

        east, north, up = local_enu_basis(lat0, lon0)

        offsets = np.arange(-swath_width_km / 2.0, swath_width_km / 2.0 + step_km, step_km)

        for x in offsets:
            p = spherical_to_cartesian(lat0, lon0, R_EARTH)
            p = p + east * x  # cross-track displacement in km

            lat, lon = cartesian_to_latlon_xyz(p)
            lon = (lon + 180.0) % 360.0 - 180.0

            rows.append({
                "orbit_name": row.get("orbit_name", "inc35_test"),
                "utc_time": row["utc_time"],
                "latitude_deg": float(lat),
                "longitude_deg": float(lon),
                "track_distance_km": float(row.get("distance_km", idx)),
                "cross_track_offset_km": float(x),
                "swath_width_km": float(swath_width_km),
                "swath_edge_flag": bool(abs(x) > 0.9 * swath_width_km / 2.0),
                "sun_elevation_deg": float(row.get("sun_elevation_deg", np.nan)),
                "sun_zenith_deg": float(row.get("sun_zenith_deg", np.nan)),
                "local_solar_time_h": float(row.get("local_solar_time_h", np.nan)),
                "local_solar_time_hhmm": row.get("local_solar_time_hhmm", ""),
                "sunlit": bool(row.get("sunlit", False)),
            })

    return pd.DataFrame(rows)

# ============================================================
# Build orbit and swath
# ============================================================

start_utc = datetime(2026, 1, 1, 0, 0, tzinfo=timezone.utc)
end_utc = start_utc + timedelta(days=1)

fine_df = generate_fine_orbit_track(
    orbit_name="inc35_test",
    altitude_km=700.0,
    inclination_deg=35.0,
    raan_deg=0.0,
    argp_deg=0.0,
    mean_anomaly_deg=0.0,
    start_utc=start_utc,
    end_utc=end_utc,
    step_minutes=2
)

track15 = resample_track_every_15km(fine_df, interval_km=15.0)
swath_df = make_swath_at_track(track15, swath_width_km=2200.0, step_km=50.0)

# Save CSV
csv_path = "osse_inc35_track_15km_swath.csv"
swath_df.to_csv(csv_path, index=False)
print(f"CSV written: {csv_path}")

# Save NetCDF
nc_path = "osse_inc35_track_15km_swath.nc"
with Dataset(nc_path, "w", format="NETCDF4") as nc:
    n = len(swath_df)

    time_dim = nc.createDimension("time", n)
    lat_var = nc.createVariable("latitude_deg", "f8", ("time",))
    lon_var = nc.createVariable("longitude_deg", "f8", ("time",))
    swath_off_var = nc.createVariable("cross_track_offset_km", "f8", ("time",))
    track_dist_var = nc.createVariable("track_distance_km", "f8", ("time",))
    swath_width_var = nc.createVariable("swath_width_km", "f8", ("time",))
    swath_edge_var = nc.createVariable("swath_edge_flag", "i1", ("time",))
    sun_el_var = nc.createVariable("sun_elevation_deg", "f8", ("time",))

    lat_var[:] = swath_df["latitude_deg"].values
    lon_var[:] = swath_df["longitude_deg"].values
    swath_off_var[:] = swath_df["cross_track_offset_km"].values
    track_dist_var[:] = swath_df["track_distance_km"].values
    swath_width_var[:] = swath_df["swath_width_km"].values
    swath_edge_var[:] = swath_df["swath_edge_flag"].astype(np.int8).values
    sun_el_var[:] = swath_df["sun_elevation_deg"].values

    nc.setncattr("orbit_name", "inc35_test")
    nc.setncattr("swath_width_km", 2200.0)
    nc.setncattr("track_sampling_km", 15.0)
    nc.setncattr("inclination_deg", 35.0)
    nc.setncattr("altitude_km", 700.0)
    nc.setncattr("start_utc", start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("end_utc", end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))

print(f"NetCDF written: {nc_path}")

# ============================================================
# Plot on Earth map with swath
# ============================================================

fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#d9d9d9", edgecolor="black", linewidth=0.4)
ax.add_feature(cfeature.OCEAN, facecolor="#cfe8ff")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.2)
ax.add_feature(cfeature.LAKES, facecolor="#cfe8ff")

# central ground track
ax.plot(track15["longitude_deg"], track15["latitude_deg"],
        transform=ccrs.PlateCarree(), color="tab:red", linewidth=2.0, label="Track (15 km)")

# swath sample points
ax.scatter(swath_df["longitude_deg"], swath_df["latitude_deg"],
           transform=ccrs.PlateCarree(), s=8, c="tab:blue", alpha=0.25, label="2200 km swath")

ax.set_title("35° Orbit Track with 2200 km Swath")
ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right")

plot_path = "osse_inc35_orbit_swath_map.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
print(f"Map plot written: {plot_path}")

print("\nFirst few swath rows:")
print(swath_df.head(10).to_string(index=False))
print(f"\nTotal swath points: {len(swath_df)}")

CSV written: osse_inc35_track_15km_swath.csv
NetCDF written: osse_inc35_track_15km_swath.nc
Map plot written: osse_inc35_orbit_swath_map.png

First few swath rows:
orbit_name             utc_time  latitude_deg  longitude_deg  track_distance_km  cross_track_offset_km  swath_width_km  swath_edge_flag  sun_elevation_deg  sun_zenith_deg  local_solar_time_h local_solar_time_hhmm  sunlit
inc35_test 2026-01-01T00:00:00Z           0.0      90.875649                0.0                -1100.0          2200.0             True         -89.991169      179.991169            6.710724                 06:43   False
inc35_test 2026-01-01T00:00:00Z           0.0      91.312398                0.0                -1050.0          2200.0             True         -89.991169      179.991169            6.710724                 06:43   False
inc35_test 2026-01-01T00:00:00Z           0.0      91.750246                0.0                -1000.0          2200.0             True         -89.991169      179.991169   

In [7]:
!apt-get update
!apt-get install -y ffmpeg

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,089 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,550 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB

In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, PillowWriter
from netCDF4 import Dataset

# ============================================================
# Generate orbit track and create animated Earth-map movie
# ============================================================

R_EARTH = 6378.137
MU_EARTH = 398600.4418

def deg2rad(x):
    return np.deg2rad(x)

def norm(v):
    return np.linalg.norm(v)

def rotation_z(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ])

def rotation_x(angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0,  c, -s],
        [0.0,  s,  c]
    ])

def spherical_to_cartesian(lat_deg, lon_deg, radius_km=R_EARTH):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    x = radius_km * np.cos(lat) * np.cos(lon)
    y = radius_km * np.cos(lat) * np.sin(lon)
    z = radius_km * np.sin(lat)
    return np.array([x, y, z])

def cartesian_to_latlon_xyz(xyz):
    x, y, z = xyz
    r = norm(xyz)
    if r == 0:
        return 0.0, 0.0
    lat = np.degrees(np.arcsin(z / r))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon

def jd_from_datetime(dt):
    epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
    return 2440587.5 + (dt - epoch).total_seconds() / 86400.0

def gmst_deg_utc(dt):
    jd = jd_from_datetime(dt)
    T = (jd - 2451545.0) / 36525.0
    gmst = (
        280.46061837
        + 360.98564736629 * (jd - 2451545.0)
        + 0.000387933 * T**2
        - (T**3) / 38710000.0
    )
    return np.mod(gmst, 360.0)

def solar_vector_eci(dt):
    jd = jd_from_datetime(dt)
    n = jd - 2451545.0
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360.0)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360.0)
    lam = L + np.deg2rad(1.915 * np.sin(g) + 0.020 * np.sin(2.0 * g))
    eps = np.deg2rad(23.439 - 0.0000004 * n)

    x = np.cos(lam)
    y = np.cos(eps) * np.sin(lam)
    z = np.sin(eps) * np.sin(lam)
    return np.array([x, y, z])

def local_enu_basis(lat_deg, lon_deg):
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)

    east = np.array([-np.sin(lon), np.cos(lon), 0.0])
    north = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
        np.cos(lat)
    ])
    up = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])
    return east, north, up

def to_enu(vector, lat_deg, lon_deg):
    east, north, up = local_enu_basis(lat_deg, lon_deg)
    e = np.dot(east, vector)
    n = np.dot(north, vector)
    u = np.dot(up, vector)
    return np.array([e, n, u])

def azimuth_from_enu(enu_vec):
    e, n, u = enu_vec
    az = np.degrees(np.arctan2(e, n))
    if az < 0:
        az += 360.0
    return az

def elevation_from_enu(enu_vec):
    e, n, u = enu_vec
    r = norm(enu_vec)
    if r == 0:
        return 0.0
    return np.degrees(np.arcsin(u / r))

def local_solar_time_from_lon_utc(dt_utc, lon_deg):
    utc_hours = dt_utc.hour + dt_utc.minute / 60.0 + dt_utc.second / 3600.0
    lst_hours = utc_hours + lon_deg / 15.0
    return lst_hours % 24.0

def orbit_position_eci_from_t0(t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg):
    a = R_EARTH + altitude_km
    n = np.sqrt(MU_EARTH / a**3)
    M0 = np.deg2rad(mean_anomaly_deg)
    M = M0 + n * t_seconds

    x_orb = a * np.cos(M)
    y_orb = a * np.sin(M)
    z_orb = 0.0
    pos_orb = np.array([x_orb, y_orb, z_orb])

    Rz_raan = rotation_z(np.deg2rad(raan_deg))
    Rx_i = rotation_x(np.deg2rad(inclination_deg))
    Rz_argp = rotation_z(np.deg2rad(argp_deg))
    R = Rz_raan @ Rx_i @ Rz_argp
    return R @ pos_orb

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

def generate_fine_orbit_track(
    orbit_name, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg,
    start_utc, end_utc, step_minutes=2
):
    records = []
    dt = start_utc
    step_td = timedelta(minutes=step_minutes)

    while dt <= end_utc:
        gmst_deg = gmst_deg_utc(dt)
        gmst_rad = np.deg2rad(gmst_deg)

        t_seconds = (dt - start_utc).total_seconds()
        pos_eci = orbit_position_eci_from_t0(
            t_seconds, altitude_km, inclination_deg, raan_deg, argp_deg, mean_anomaly_deg
        )
        pos_ecef = rotation_z(gmst_rad) @ pos_eci

        lat_deg, lon_deg = cartesian_to_latlon_xyz(pos_ecef)
        lon_deg = (lon_deg + 180.0) % 360.0 - 180.0

        obs_ecef = spherical_to_cartesian(lat_deg, lon_deg, R_EARTH)
        sat_rel = pos_ecef - obs_ecef
        sat_enu = to_enu(sat_rel, lat_deg, lon_deg)
        sat_az = azimuth_from_enu(sat_enu)
        sat_el = elevation_from_enu(sat_enu)
        off_nadir = 90.0 - sat_el

        sun_eci = solar_vector_eci(dt)
        sun_ecef = rotation_z(gmst_rad) @ sun_eci
        sun_rel = sun_ecef - obs_ecef
        sun_enu = to_enu(sun_rel, lat_deg, lon_deg)
        sun_az = azimuth_from_enu(sun_enu)
        sun_el = elevation_from_enu(sun_enu)
        sun_zenith = 90.0 - sun_el
        sunlit = sun_el > 0.0

        local_solar_time_h = local_solar_time_from_lon_utc(dt, lon_deg)
        total_minutes = int(round(local_solar_time_h * 60.0))
        hh = total_minutes // 60
        mm = total_minutes % 60
        lst_hhmm = f"{hh:02d}:{mm:02d}"

        records.append({
            "orbit_name": orbit_name,
            "utc_time": dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "unix_seconds": int(dt.timestamp()),
            "latitude_deg": float(lat_deg),
            "longitude_deg": float(lon_deg),
            "satellite_azimuth_deg": float(sat_az),
            "satellite_elevation_deg": float(sat_el),
            "sun_azimuth_deg": float(sun_az),
            "sun_elevation_deg": float(sun_el),
            "sun_zenith_deg": float(sun_zenith),
            "local_solar_time_h": float(local_solar_time_h),
            "local_solar_time_hhmm": lst_hhmm,
            "sunlit": bool(sunlit),
            "off_nadir_angle_deg": float(off_nadir),
            "altitude_km": float(altitude_km),
            "inclination_deg": float(inclination_deg),
            "raan_deg": float(raan_deg),
            "argp_deg": float(argp_deg),
            "mean_anomaly_deg": float(mean_anomaly_deg),
        })

        dt += step_td

    return pd.DataFrame(records)

def resample_track_every_15km(df, interval_km=15.0):
    if len(df) < 2:
        return df.copy()

    cum = np.zeros(len(df))
    for i in range(1, len(df)):
        cum[i] = cum[i-1] + haversine_km(
            df["latitude_deg"].iloc[i-1],
            df["longitude_deg"].iloc[i-1],
            df["latitude_deg"].iloc[i],
            df["longitude_deg"].iloc[i]
        )

    total_dist = cum[-1]
    sample_dist = np.arange(0.0, total_dist + interval_km, interval_km)

    out = {}
    for col in df.columns:
        if col in ["latitude_deg", "longitude_deg", "satellite_azimuth_deg",
                   "satellite_elevation_deg", "sun_azimuth_deg", "sun_elevation_deg",
                   "sun_zenith_deg", "local_solar_time_h", "off_nadir_angle_deg",
                   "altitude_km", "inclination_deg", "raan_deg", "argp_deg",
                   "mean_anomaly_deg", "unix_seconds"]:
            out[col] = np.interp(sample_dist, cum, df[col].values)

    if "utc_time" in df.columns:
        out["utc_time"] = [df["utc_time"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist]
    if "local_solar_time_hhmm" in df.columns:
        out["local_solar_time_hhmm"] = [df["local_solar_time_hhmm"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist]
    if "sunlit" in df.columns:
        out["sunlit"] = np.array([df["sunlit"].iloc[np.argmin(np.abs(cum - d))] for d in sample_dist], dtype=bool)
    if "orbit_name" in df.columns:
        out["orbit_name"] = [df["orbit_name"].iloc[0]] * len(sample_dist)

    out["distance_km"] = sample_dist
    return pd.DataFrame(out)

# ------------------------------------------------------------
# Generate orbit points
# ------------------------------------------------------------
start_utc = datetime(2026, 1, 1, 0, 0, tzinfo=timezone.utc)
end_utc = start_utc + timedelta(days=1)

fine_df = generate_fine_orbit_track(
    orbit_name="inc35_test",
    altitude_km=700.0,
    inclination_deg=35.0,
    raan_deg=0.0,
    argp_deg=0.0,
    mean_anomaly_deg=0.0,
    start_utc=start_utc,
    end_utc=end_utc,
    step_minutes=2
)

track15 = resample_track_every_15km(fine_df, interval_km=15.0)

# Save CSV and NetCDF
track15.to_csv("orbit_track_15km.csv", index=False)

with Dataset("orbit_track_15km.nc", "w", format="NETCDF4") as nc:
    n = len(track15)
    dim = nc.createDimension("time", n)

    lat = nc.createVariable("latitude_deg", "f8", ("time",))
    lon = nc.createVariable("longitude_deg", "f8", ("time",))
    dist = nc.createVariable("distance_km", "f8", ("time",))
    sun_elev = nc.createVariable("sun_elevation_deg", "f8", ("time",))

    lat[:] = track15["latitude_deg"].values
    lon[:] = track15["longitude_deg"].values
    dist[:] = track15["distance_km"].values
    sun_elev[:] = track15["sun_elevation_deg"].values

    nc.setncattr("orbit_name", "inc35_test")
    nc.setncattr("track_sampling_km", 15.0)
    nc.setncattr("inclination_deg", 35.0)
    nc.setncattr("start_utc", start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))
    nc.setncattr("end_utc", end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"))

# ------------------------------------------------------------
# Build animation
# ------------------------------------------------------------
fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#d9d9d9", edgecolor="black", linewidth=0.3)
ax.add_feature(cfeature.OCEAN, facecolor="#cfe8ff")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.2)
ax.add_feature(cfeature.LAKES, facecolor="#cfe8ff", edgecolor="none")

lon = track15["longitude_deg"].values
lat = track15["latitude_deg"].values

line, = ax.plot([], [], transform=ccrs.PlateCarree(), color="tab:red", linewidth=2.0)
dot, = ax.plot([], [], "o", transform=ccrs.PlateCarree(), color="tab:red", markersize=6)

# full path in faint blue
ax.plot(lon, lat, transform=ccrs.PlateCarree(), color="tab:blue", linewidth=1.0, alpha=0.4)

ax.set_title("35° Orbit Track on Earth Map")
ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)

def init():
    line.set_data([], [])
    dot.set_data([], [])
    return line, dot

def update(frame):
    # show only the path up to the current frame
    n = frame + 1
    line.set_data(lon[:n], lat[:n])
    dot.set_data([lon[n-1]], [lat[n-1]])
    return line, dot

ani = FuncAnimation(
    fig,
    update,
    frames=np.arange(0, len(track15)),
    init_func=init,
    blit=True,
    interval=40
)

video_path = "orbit_animation.mp4"
ani.save(video_path, writer="ffmpeg", fps=20)
print(f"Video saved: {video_path}")

# Optional save a static map snapshot
static_path = "orbit_static_map.png"
plt.savefig(static_path, dpi=200, bbox_inches="tight")
print(f"Static image saved: {static_path}")

# Download files from Colab
from google.colab import files
files.download(video_path)
files.download(static_path)
files.download("orbit_track_15km.csv")
files.download("orbit_track_15km.nc")

CalledProcessError: Command '['ffmpeg', '-f', 'rawvideo', '-vcodec', 'rawvideo', '-s', '1200x700', '-pix_fmt', 'rgba', '-framerate', '20', '-loglevel', 'error', '-i', 'pipe:', '-vcodec', 'h264', '-pix_fmt', 'yuv420p', '-y', 'orbit_animation.mp4']' returned non-zero exit status 255.

In [3]:
!pip install numpy pandas netCDF4 matplotlib cartopy

In [ ]:
!python simulation.py